# N099 · 一维DP与以当前位置结尾的状态

**目标：** 区分前缀最优和必须选当前位置的最优。

**先修：** N098, N015。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 线性递推；House Robber；Kadane；滚动变量；负数基例。

## 从问题到算法

打家劫舍的状态可压缩为两个前缀最优值：不选末屋沿用前一项，选末屋则只能接前两项。环形问题把首尾冲突拆成“不含首屋”与“不含末屋”。最大子数组不同：必须保留“以当前位置结尾”的非空状态，不能将全负输入错误地返回0。

## 最小实现

**本章接口：** `rob(nums)`、`rob_circular(nums)`、`max_subarray(nums)`

In [1]:
def rob(nums):
    before=prev=0
    for x in nums: before,prev=prev,max(prev,before+x)
    return prev

def rob_circular(nums):
    if len(nums)<=1: return max([0]+list(nums))
    return max(rob(nums[:-1]),rob(nums[1:]))

def max_subarray(nums):
    if not nums: raise ValueError('nonempty array required')
    ending=best=nums[0]
    for x in nums[1:]: ending=max(x,ending+x); best=max(best,ending)
    return best

## 正确性、前提与复杂度

所有合法独立集按是否选择末项划分，取两者最大。环上合法集必不同时包含首尾，两个线性子问题覆盖全部可能。非空连续区间若以i结尾，要么从i开始，要么接在以i−1结尾的最优区间后。

**代价：** O(n)时间、线性rob与最大子数组O(1)辅助；环形实现切片O(n)辅助，可改索引范围消除。rob允许不选任何项；max_subarray必须非空。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[-2,1,-3,4,-1,2,1,-5,4]; ending=best=a[0]; rows=[(0,a[0],ending,best)]
for i,x in enumerate(a[1:],1): ending=max(x,ending+x); best=max(best,ending); rows.append((i,x,ending,best))
show_table(['位置','值','以此结尾','全局最优'],rows)

位置,值,以此结尾,全局最优
0,-2,-2,-2
1,1,1,1
2,-3,-2,1
3,4,4,4
4,-1,3,4
5,2,5,5
6,1,6,6
7,-5,1,6
8,4,5,6


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert rob([2,7,9,3,1])==12 and rob_circular([2,3,2])==3
assert max_subarray([-3,-1,-2])==-1
from itertools import product
for n in range(1,7):
    for a in product((-2,0,3),repeat=n):
        linear=[]; circular=[]
        for mask in range(1<<n):
            if mask&(mask<<1): continue
            value=sum(a[i] for i in range(n) if mask>>i&1); linear.append(value)
            if n==1 or not (mask&1 and mask>>(n-1)&1): circular.append(value)
        assert rob(a)==max(linear) and rob_circular(a)==max(circular)
        assert max_subarray(a)==max(sum(a[l:r]) for l in range(n) for r in range(l+1,n+1))
print('N099: 所有本章断言通过')

N099: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 把环拆成互斥首尾约束。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 返回一条最优选择方案。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 198. House Robber（canonical）：[官方入口](https://leetcode.com/problems/house-robber/)
- 213. House Robber II（canonical）
- 53. Maximum Subarray（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。